# 🔬 Arm 5: SEGO-GRPO (Syntactic-Execution Gated Policy Optimization)
**Project:** Diagnosing and Resolving Code SLM Mimicry  
**Authors:** Omar Abdelhamid, Nour Walid  
**Supervisor:** Dr. Ghada Soliman  
**Organization:** Orange Innovation Labs — AI Research & Advanced Innovation Division  

---

## 🎯 Research Motivation & Scientific Signature
In our empirical evaluation across 7 models (5,348 benchmark runs):
- **M4 Standard GRPO** suffered from **Reasoning Bloat / Overthinking Tax = 2.724** (length hacking).
- **Yeo et al. (2025)** proved that naive negative length penalties cause 'fast and wrong' answers (premature disengagement).
- **LUSPO (2025)** & **GRPO-LEAD (2025)** treat code as flat word tokens, failing to capture true algorithmic complexity.

### The Unified SEGO Formulation (Execution-Gated Structural Optimization):
$$\mathcal{R}_{\text{SEGO}}(y, x) = \mathcal{R}_{\text{step}}(y) \cdot \Big[ 1 + \alpha \cdot \text{sim}_{\text{AST}}(y, y^*) - \gamma \cdot \Omega_{\text{AST}}(y, y^*) \Big] - \lambda \cdot \mathcal{L}_{\text{inv}}(x, x')$$

| Component | Academic Grounding | Novel Mechanism |
|---|---|---|
| **Execution Gate ($\mathcal{R}_{\text{step}}$)** | Yeo et al. (2025), CodePRM (ACL 2025) | Zeros penalties on failure; allows unhindered exploration |
| **AST Tree Fidelity ($\text{sim}_{\text{AST}}$)** | TreeDiff (ASE 2025), VeriSeek (ICSE 2025) | Inductive bias toward canonical control flow |
| **Syntactic Tree Bloat ($\Omega_{\text{AST}}$)** | **Our Primary Novelty** | Penalizes AST node explosion rather than surface characters |
| **Invariance Regularizer ($\mathcal{L}_{\text{inv}}$)** | Inv-GRPO (Aly et al. 2026) | Enforces behavioral stability under surface perturbations |

In [1]:
# ── 1. Setup Environment & Path Resolution ────────────────────────────────────
import os, sys, subprocess
from pathlib import Path

if os.path.exists('/kaggle'):
    print('⚡ Running inside Kaggle environment.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes', 'accelerate', 'peft'])
    os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
    if os.path.exists('/kaggle/working/reo/src'):
        REPO_ROOT = '/kaggle/working/reo'
        subprocess.run(['git', '-C', REPO_ROOT, 'pull', 'origin', 'main'])
    else:
        subprocess.run(['git', 'clone', 'https://github.com/OmarAbdelhamidAly/diagnosing-resolving-code-slm-mimicry.git', '/kaggle/working/reo'])
        REPO_ROOT = '/kaggle/working/reo'
else:
    _nb_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
    REPO_ROOT = str(Path(_nb_dir).resolve().parents[0]) if Path(_nb_dir).name == 'notebooks' else str(Path(_nb_dir).resolve())

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print(f'Working directory set to: {os.getcwd()}')

# Check PyTorch & GPU
import torch
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device      : {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)')


Working directory set to: C:\Users\Lenovo\Downloads\Reasoning\reo
PyTorch Version : 2.6.0+cu124
CUDA Available  : True
GPU Device      : NVIDIA GeForce RTX 3070 Ti Laptop GPU (8.59 GB)


In [2]:
# ── 2. Load Multi-Rung Training Pool ──────────────────────────────────────────
import os, json
from pathlib import Path
from src.evaluation.registry import BenchmarkRegistry

data_dir = Path(REPO_ROOT) / 'data' / 'ladder'
if not (data_dir / 'L0_humaneval_standard.jsonl').exists() and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'L0_humaneval_standard.jsonl' in files or 'L1_evoeval_subtle.jsonl' in files:
            data_dir = Path(root)
            break

print(f'Loading tasks from: {data_dir}')
registry = BenchmarkRegistry(data_dir=data_dir)

# Multi-rung pool with canonical solutions: L1, L3, L4, L5
tasks = [
    {
        'prompt': t.prompt,
        'test': t.test,
        'entry_point': t.entry_point,
        'canonical_solution': t.canonical_solution or '',
    }
    for t in (
        registry.get_tasks('L1')
        + registry.get_tasks('L3')
        + registry.get_tasks('L4')
        + registry.get_tasks('L5')
    )
]

# ── Self-Healing Fallback for Kaggle without attached input dataset ──
if len(tasks) == 0:
    print('⚠️ No ladder files found locally or in /kaggle/input (no dataset attached).')
    print('🌐 Automatically downloading EvoEval multi-rung training pool from Hugging Face...')
    from datasets import load_dataset
    hf_splits = {
        'L1': ('EvoEval_subtle', 'subtle'),
        'L3': ('EvoEval_creative', 'creative'),
        'L4': ('EvoEval_difficult', 'difficult'),
        'L5': ('EvoEval_combine', 'combine'),
    }
    tasks = []
    out_ladder = Path(REPO_ROOT) / 'data' / 'ladder'
    out_ladder.mkdir(parents=True, exist_ok=True)
    for level, (hf_name, suffix) in hf_splits.items():
        print(f'   Fetching EvoEval/{hf_name} from Hugging Face...')
        ds = load_dataset(f'EvoEval/{hf_name}', split='test')
        records = []
        for item in ds:
            t_dict = {
                'task_id': item['task_id'],
                'ladder_level': level,
                'benchmark': hf_name,
                'prompt': item['prompt'],
                'canonical_solution': item.get('canonical_solution', '') or '',
                'test': item['test'],
                'entry_point': item['entry_point'],
            }
            records.append(t_dict)
            tasks.append({
                'prompt': t_dict['prompt'],
                'test': t_dict['test'],
                'entry_point': t_dict['entry_point'],
                'canonical_solution': t_dict['canonical_solution'],
            })
        cache_file = out_ladder / f'{level}_evoeval_{suffix}.jsonl'
        with open(cache_file, 'w', encoding='utf-8') as cf:
            for r in records:
                cf.write(json.dumps(r, ensure_ascii=False) + '\n')

print(f'✅ Successfully loaded {len(tasks)} tasks for S³-GRPO training.')


c:\Users\Lenovo\anaconda3\envs\unsloth_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Loading tasks from: C:\Users\Lenovo\Downloads\Reasoning\reo\data\ladder
✅ Successfully loaded 400 tasks for S³-GRPO training.


In [3]:
# ── 3. Dry-Run Verification of SEGO Reward Engine ─────────────────────────
from src.arms.arm5_hybrid_s3.reward_engine import S3RewardEngine

engine = S3RewardEngine(
    w_step=0.50,
    w_ast=0.30,
    w_inv=0.20,
    w_parsimony=0.20,  # gamma: AST tree bloat weight
    sandbox_timeout=3.0,
)

sample_task = tasks[0]
print('Evaluating sample task with canonical reference solution:')
print('Prompt sample:', sample_task['prompt'][:100], '...')

res = engine.compute_reward(
    prompt=sample_task['prompt'],
    solution=sample_task['canonical_solution'],
    test=sample_task['test'],
    entry_point=sample_task['entry_point'],
    canonical_solution=sample_task['canonical_solution'],
)
print('SEGO Reward Breakdown:', res)
print(f"AST Nodes: Sol={res['nodes_sol']}, Ref={res['nodes_ref']} | Bloat Omega_AST={res['omega_ast']}")
assert res['all_passed'] is True
print('✅ SEGO Reward Engine Verified: All assertions passed cleanly without tree bloat.')


Evaluating sample task with canonical reference solution:
Prompt sample: 
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
  ...
SEGO Reward Breakdown: {'r_total': 1.5, 'r_step': 1.0, 'r_ast': 1.0, 'l_inv': 0.0, 'omega_ast': 0.0, 'nodes_sol': 44, 'nodes_ref': 44, 'all_passed': True}
AST Nodes: Sol=44, Ref=44 | Bloat Omega_AST=0.0
✅ SEGO Reward Engine Verified: All assertions passed cleanly without tree bloat.


In [4]:
# ── 4. Execute 5,000-Step S³-GRPO Training Loop ─────────────────────────────
from src.arms.arm5_hybrid_s3.trainer import S3GRPOTrainer

OUTPUT_DIR = "checkpoints/s3_grpo_final"
trainer = S3GRPOTrainer(
    output_dir=OUTPUT_DIR,
    group_size=4,
    learning_rate=1e-5,
    max_new_tokens=384,
    w_step=0.50,
    w_ast=0.30,
    w_inv=0.20,
    w_parsimony=0.20,
)

# 5,000 steps production run on Kaggle GPU (~3.5-4h on T4/P100)
NUM_STEPS = 5000
SAVE_STEPS = 500

print(f"🚀 Starting S³-GRPO Flagship Optimization ({NUM_STEPS} steps, G=4)...")
history = trainer.train(
    tasks=tasks,
    num_steps=NUM_STEPS,
    grad_accum_steps=2,
    save_steps=SAVE_STEPS,
)
print("🎉 Training Completed Successfully! Final Checkpoint saved to:", OUTPUT_DIR)


c:\Users\Lenovo\anaconda3\envs\unsloth_env\lib\site-packages\transformers\utils\hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


[S³-GRPO] Loading tokenizer for: Qwen/Qwen2.5-Coder-1.5B-Instruct
[S³-GRPO] Loading 4-bit NF4 quantized model...


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


[S³-GRPO] LoRA initialized: 18,464,768 trainable params (2.04%).
🚀 Starting S³-GRPO Optimization (500 steps)...

[S³-GRPO] 🚀 Starting S³-GRPO Multi-Objective Training (500 steps, G=4)...


S³-GRPO Training:   5%|▌         | 26/500 [52:47<16:02:17, 121.81s/it, R=0.83, StepR=0.75, AST=0.69, Tax=0.49, Pass%=75%, Loss=-0.020] 


KeyboardInterrupt: 

In [ ]:
# ── 5. Plot SEGO Training Dynamics ────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(history['step'], history['loss'], color='#E74C3C', lw=1.5)
axes[0].set_title('Policy Loss (Surrogate Objective)')
axes[0].set_xlabel('Step')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['step'], history['mean_reward'], color='#2ECC71', lw=1.5, label='SEGO Reward')
axes[1].plot(history['step'], history['mean_step_reward'], color='#3498DB', lw=1.2, ls='--', label='Stepwise R')
axes[1].set_title('SEGO & Process Rewards')
axes[1].set_xlabel('Step')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['step'], history['mean_ast_sim'], color='#9B59B6', lw=1.5, label='AST Tree Sim')
axes[2].plot(history['step'], history['mean_parsimony_penalty'], color='#F39C12', lw=1.2, ls=':', label='AST Tree Bloat (Ω_AST)')
axes[2].set_title('Syntactic Tree Dynamics (SEGO)')
axes[2].set_xlabel('Step')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/s3_grpo_training_dynamics.png', dpi=300)
plt.show()
print('💾 Saved SEGO training dynamics plot to results/s3_grpo_training_dynamics.png')


## 📊 6. Benchmark Evaluation Across Reduction Ladder & LiveCodeBench (Lite & Full)
Evaluate the newly trained **SEGO-GRPO (M8)** model across all ladder levels ($L_0$ to $L_5$) and both **LiveCodeBench Lite** (100 tasks) and **LiveCodeBench Full** (1,055 tasks).


In [ ]:
# ── 6. Benchmark Evaluation of SEGO-GRPO (M8) ──────────────────────────────
import os
from pathlib import Path
from src.evaluation.registry import BenchmarkRegistry
from src.evaluation.suite import EvaluationSuite

print("=" * 75)
print("🔍 Evaluating SEGO-GRPO (M8 Flagship) on Reduction Ladder & LiveCodeBench")
print("=" * 75)

# Ensure full LiveCodeBench is available locally for evaluation
lcb_full_path = Path(REPO_ROOT) / "data" / "ladder" / "Ctrl_livecode_full.jsonl"
if not lcb_full_path.exists():
    print("🌐 Full LiveCodeBench benchmark not detected. Streaming (1,055 tasks, ~20s)...")
    try:
        from scripts.download_full_livecodebench import download_and_format_full_lcb
        download_and_format_full_lcb(out_path=str(lcb_full_path))
    except Exception as e:
        print(f"⚠️ Could not stream full LCB: {e}")

eval_registry = BenchmarkRegistry(data_dir=data_dir)
suite = EvaluationSuite(registry=eval_registry)

# Run evaluation on trained checkpoint across all available levels (L0-L5, Ctrl, Ctrl_Full)
sego_report = suite.evaluate_model(
    model_id="M8_sego_grpo",
    model_name_or_path=trainer.model_name,
    adapter_path=OUTPUT_DIR,
    output_dir="results/M8_sego_grpo",
    max_tasks_per_level=None,  # evaluate full benchmark
)

print("\n✅ Evaluation of M8 SEGO-GRPO Completed!")
print(f"   Ladder AUC  : {sego_report.get('ladder_auc', 0.0) * 100:.1f}%")
print(f"   LCB (Lite)  : {sego_report.get('ood_score', 0.0) * 100:.1f}% (100 tasks)")
if sego_report.get("ood_full_score") is not None:
    print(f"   LCB (Full)  : {sego_report.get('ood_full_score', 0.0) * 100:.1f}% (1,055 tasks)")


## 🏆 7. Comparative Benchmark Leaderboard (M1 Baseline → M8 SEGO-GRPO)
Direct comparison between SEGO-GRPO and all historical research arms (M1 to M7) across both Lite & Full OOD benchmarks.


In [ ]:
# ── 7. Multi-Model Benchmark Comparison Table & Curves ────────────────────
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

# 1. Load historical master comparison table from repo
csv_path = Path(REPO_ROOT) / "results" / "master_comparison_table.csv"
df_master = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()

# 2. Extract SEGO-GRPO (M8) metrics
sego_p1 = sego_report.get("pass_at_1", {})
m1_auc = 0.774  # M1_baseline AUC reference

m8_auc = sego_report.get("ladder_auc", 0.0)
m8_rel = ((m8_auc - m1_auc) / m1_auc) * 100 if m1_auc > 0 else 0.0

m8_row = {
    "Model": "M8_sego_grpo (Ours)",
    "L0": f"{sego_p1.get('L0', 0.0)*100:.1f}%",
    "L1": f"{sego_p1.get('L1', 0.0)*100:.1f}%",
    "L2": f"{sego_p1.get('L2', 0.0)*100:.1f}%",
    "L3": f"{sego_p1.get('L3', 0.0)*100:.1f}%",
    "L4": f"{sego_p1.get('L4', 0.0)*100:.1f}%",
    "L5": f"{sego_p1.get('L5', 0.0)*100:.1f}%",
    "Ctrl (Lite)": f"{sego_report.get('ood_score', 0.0)*100:.1f}%",
    "Ctrl (Full)": f"{sego_report.get('ood_full_score', 0.0)*100:.1f}%" if sego_report.get("ood_full_score") is not None else "N/A",
    "Ladder AUC": f"{m8_auc*100:.1f}%",
    "Degrad. Slope": f"{sego_report.get('degradation_slope', 0.0):+.3f}",
    "Collapse Point": sego_report.get("collapse_point", "None"),
    "Consistency Δ": f"{sego_report.get('consistency_delta', 0.0)*100:.1f}%",
    "Overthinking Tax": f"{sego_report.get('overthinking_tax', 0.0):.3f}",
    "Rel. Gain vs M1": f"{m8_rel:+.1f}%",
}

# Align Ctrl column name in df_master if needed
if "Ctrl (OOD)" in df_master.columns and "Ctrl (Lite)" not in df_master.columns:
    df_master = df_master.rename(columns={"Ctrl (OOD)": "Ctrl (Lite)"})

# Append M8 to master leaderboard
if not df_master.empty and "Model" in df_master.columns:
    df_updated = df_master[~df_master["Model"].str.contains("M8", na=False)].copy()
    df_updated = pd.concat([df_updated, pd.DataFrame([m8_row])], ignore_index=True)
else:
    df_updated = pd.DataFrame([m8_row])

print("=" * 110)
print("🏆 COMPLETE BENCHMARK LEADERBOARD (M1 BASELINE TO M8 SEGO-GRPO)")
print("=" * 110)
display(df_updated)

# Save updated master table to results
out_csv = Path(REPO_ROOT) / "results" / "master_comparison_with_m8.csv"
df_updated.to_csv(out_csv, index=False)
print(f"\n💾 Saved updated comparison to: {out_csv}")

# 3. Plot Pass@1 Degradation Curves (M1 vs M4 vs M6 vs M7 vs M8)
fig, ax = plt.subplots(figsize=(12, 6), dpi=200)
levels = ["L0", "L1", "L2", "L3", "L4", "L5", "Ctrl (Lite)"]
x_pos = range(len(levels))

curve_models = {
    "M1_baseline": ("#7F8C8D", "M1 Baseline (Raw Qwen2.5-Coder)", ":"),
    "M4_standard_grpo": ("#E74C3C", "M4 Standard GRPO", "--"),
    "M6_inv_grpo": ("#3498DB", "M6 Invariant GRPO", "-."),
    "M7_step_rlvr": ("#F39C12", "M7 Step-RLVR", "-."),
    "M8_sego_grpo": ("#2ECC71", "M8 SEGO-GRPO (Flagship Ours)", "-"),
}

ax.axhline(0.50, color="#E74C3C", linestyle=":", alpha=0.6, label="Collapse Threshold (50%)")
ax.axvspan(5.5, 6.5, alpha=0.08, color="#9B59B6", label="LiveCodeBench (OOD)")

for mid_key, (color, label, ls) in curve_models.items():
    row_match = df_updated[df_updated["Model"].str.contains(mid_key, na=False)]
    if not row_match.empty:
        vals = [
            float(str(row_match.iloc[0].get(lvl, 0.0)).replace("%", "")) / 100.0
            if str(row_match.iloc[0].get(lvl, 0.0)) != "N/A" else 0.0
            for lvl in levels
        ]
        lw = 3.2 if "M8" in mid_key else 1.8
        ax.plot(x_pos, vals, marker="o", color=color, label=label,
                linewidth=lw, linestyle=ls, markersize=8 if "M8" in mid_key else 5,
                zorder=5 if "M8" in mid_key else 3)

ax.set_xticks(x_pos)
ax.set_xticklabels(["L0\nStd", "L1\nSubtle", "L2\nTool", "L3\nCreative", "L4\nDiff", "L5\nComb", "Ctrl\nLCB Lite"], fontsize=10)
ax.set_ylim(-0.02, 1.05)
ax.set_xlabel("Reduction Ladder Level", fontsize=11, fontweight="bold")
ax.set_ylabel("Pass@1 Accuracy", fontsize=11, fontweight="bold")
ax.set_title("Reduction Ladder: Anti-Mimicry Generalization Across All Arms", fontsize=13, pad=12, fontweight="bold")
ax.grid(True, linestyle=":", alpha=0.5)
ax.legend(frameon=True, loc="lower left", fontsize=9)
plt.tight_layout()
plt.savefig("results/sego_grpo_comparison_curves.png", dpi=300)
plt.show()
print("💾 Saved comparison degradation curves to results/sego_grpo_comparison_curves.png")
